# Gap 2: zero-shot generalization to larger glasses and new quench histories

**Kaggle settings:** GPU T4 ×2 (one GPU also works), Internet On, and notebook access to the **GITHUB_TOKEN** secret with write permission to `chilalaabubakar/IDL-PROJECT`. Then **Run All**.

This is the dedicated gap-2 notebook. It runs six conditions: 256 and 1024 atoms crossed with cooling rates 0.001, 0.01 and 0.1. It generates 100 fresh hosts per condition and saves 256 matched requests; evaluates all three retained independent training seeds without retraining; and runs a same-checkpoint seed-0 sham and matched hand-insertion baseline from the original training library. Detector thresholds, guidance 2, radius 6 and the 100+100-step sampler are frozen.

**Expect multiple Kaggle sessions.** Full GPU sampling runtime is not yet benchmarked; this study is substantially larger than gap 1. The notebook saves valid per-host candidates and completed sampling batches to GitHub every five minutes. It stops after an 11-hour budget and pushes an `incomplete` status. Re-run this notebook in a fresh session to restore and continue the most recent incomplete run. Keep batch size 4 and the fixed settings unchanged. Work from the two GPUs can finish at different times; the slower task determines each stage's completion.

Production results appear under `results/kaggle_gap2_generalization_<UTC timestamp>/`. Only `progress.json: complete` together with `study_status.json: complete` means all six conditions finished. Earlier completed conditions remain useful while later ones run. Raw logs stay under `/kaggle/working/gap2-generalization/`; save Kaggle output files when diagnosing a failure.

This notebook measures **placement generalization**, convergence, runtime and paired baseline differences. It does not automatically establish natural structural equivalence or thermal retention on the new conditions; those require additional analyses/MD.


In [ ]:
import os, sys, subprocess, time, json, shutil, signal, tempfile, zipfile
from pathlib import Path
from datetime import datetime, timezone
from kaggle_secrets import UserSecretsClient

REPO = "https://github.com/chilalaabubakar/IDL-PROJECT.git"
ROOT = Path("/kaggle/working/IDL-PROJECT-gap2-code")
WORK = Path("/kaggle/working/gap2-generalization")
PUSHROOT = Path("/kaggle/working/gap2-publish")
TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
assert TOKEN, "Enable the GITHUB_TOKEN Kaggle secret"

def run(args, cwd=None, env=None):
    result = subprocess.run(list(map(str, args)), cwd=cwd, env=env,
                            text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(result.stderr.replace(TOKEN, "[REDACTED]")[-4000:])
    return result.stdout

if not ROOT.exists():
    run(["git", "clone", REPO, ROOT])
else:
    assert not run(["git", "status", "--porcelain"], ROOT).strip(), "Checkout has unsaved changes"
    run(["git", "pull", "--ff-only", "origin", "main"], ROOT)
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[md]"], ROOT)
import torch
import numpy as np
assert torch.cuda.is_available(), "Select a GPU accelerator in Kaggle settings"
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
CODE_SHA = run(["git", "rev-parse", "HEAD"], ROOT).strip()
ENV = dict(os.environ, OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")
# Discover the installed wheel location: Kaggle's pip prefix can differ from sys.prefix.
from importlib.metadata import distribution
mpi_dist = distribution("mpich")
mpi_dirs = sorted({str(mpi_dist.locate_file(f).resolve().parent)
                   for f in (mpi_dist.files or [])
                   if f.name == "libmpi.so.12" and mpi_dist.locate_file(f).exists()})
assert mpi_dirs, "The mpich wheel did not install libmpi.so.12; reinstall mpich before continuing"
ENV["LD_LIBRARY_PATH"] = ":".join(mpi_dirs + [str(Path(sys.prefix)/"lib"), ENV.get("LD_LIBRARY_PATH", "")])
# Verify the MD wheel and its MPI dependency before starting costly generation.
run([sys.executable, "-c", "from lammps import lammps; x=lammps(cmdargs=['-screen','none','-log','none']); x.close()"], ROOT, ENV)
print("Code revision:", CODE_SHA)


In [ ]:
# Resume a previous incomplete run, including a fresh Kaggle session.
if not PUSHROOT.exists():
    run(["git", "clone", REPO, PUSHROOT])
else:
    assert not run(["git", "status", "--porcelain"], PUSHROOT).strip(), "Publisher has unsaved changes"
    run(["git", "pull", "--ff-only", "origin", "main"], PUSHROOT)
prior = sorted((PUSHROOT / "results").glob("kaggle_gap2_generalization_*"))
resume = next((p for p in reversed(prior) if (p/"progress.json").exists()
               and json.loads((p/"progress.json").read_text()).get("status") != "complete"), None)
if not WORK.exists():
    if resume:
        shutil.copytree(resume, WORK)
    else:
        WORK.mkdir(parents=True)
run_name = resume.name if resume else "kaggle_gap2_generalization_" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%SZ")
if resume:
    saved_sha = json.loads((resume/"progress.json").read_text())["code_revision"]
    run(["git", "checkout", "--detach", saved_sha], ROOT)
    CODE_SHA = saved_sha
PUBLISH = PUSHROOT / "results" / run_name
PUBLISH.mkdir(parents=True, exist_ok=True)
START = time.monotonic()
STATUS = "running"

# Git credentials are supplied only through a temporary askpass environment.
# They are never saved to the repository, remote URL or notebook outputs.
def push():
    for src in WORK.rglob("*"):
        if not src.is_file() or src.suffix not in {".npz", ".json", ".csv", ".md"}:
            continue
        try:
            if src.suffix == ".npz":
                with np.load(src) as data:
                    for key in data.files:
                        data[key]  # force decompression/CRC check; skip partial writes
            elif src.suffix == ".json":
                json.loads(src.read_text())
        except (OSError, ValueError, EOFError, zipfile.BadZipFile):
            continue
        dest = PUBLISH / src.relative_to(WORK)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dest)
    progress = dict(status=STATUS, code_revision=CODE_SHA, torch=torch.__version__,
                    cuda=torch.version.cuda, batch_size=4, n_requests_per_model=256, n_hosts_per_condition=100,
                    updated_utc=datetime.now(timezone.utc).isoformat())
    (PUBLISH/"progress.json").write_text(json.dumps(progress, indent=2)+"\n")
    (WORK/"progress.json").write_text(json.dumps(progress, indent=2)+"\n")
    run(["git", "config", "user.name", "Gap 2 generalization"], PUSHROOT)
    run(["git", "config", "user.email", "gap2-evaluation@users.noreply.github.com"], PUSHROOT)
    run(["git", "add", str(PUBLISH.relative_to(PUSHROOT))], PUSHROOT)
    if not run(["git", "diff", "--cached", "--name-only"], PUSHROOT).strip():
        return
    run(["git", "commit", "-m", f"Save Gap 2 generalization checkpoints ({STATUS})"], PUSHROOT)
    with tempfile.TemporaryDirectory() as td:
        helper=Path(td)/"askpass.py"
        helper.write_text("#!/usr/bin/env python3\nimport os,sys\nprint('x-access-token' if 'Username' in sys.argv[1] else os.environ['GAP2_GIT_TOKEN'])\n")
        helper.chmod(0o700)
        auth=dict(os.environ, GIT_ASKPASS=str(helper), GIT_TERMINAL_PROMPT="0", GAP2_GIT_TOKEN=TOKEN)
        for attempt in range(3):
            # Merge preserves the exact measured code revision and concurrent research updates.
            run(["git", "fetch", "origin", "main"], PUSHROOT, auth)
            run(["git", "merge", "--no-edit", "origin/main"], PUSHROOT, auth)
            result=subprocess.run(["git", "push", "origin", "HEAD:main"], cwd=PUSHROOT,
                                  env=auth, text=True, capture_output=True)
            if result.returncode == 0:
                print("Saved:", run_name, STATUS, flush=True)
                return
        raise RuntimeError("GitHub checkpoint push failed: " + result.stderr.replace(TOKEN,"[REDACTED]")[-2000:])
push()  # Verify write access before spending GPU time.
print("Result location:", "https://github.com/chilalaabubakar/IDL-PROJECT/tree/main/results/"+run_name)


In [ ]:
# Child processes write logs to files so notebook output cannot stall sampling.
def monitor(commands):
    processes=[]
    handles=[]
    last_push=time.monotonic()
    last_report=0
    try:
        for name, args, env in commands:
            handle=(WORK/(name+".log")).open("a")
            handles.append(handle)
            processes.append(subprocess.Popen(args, cwd=ROOT, env=env, stdout=handle,
                                               stderr=subprocess.STDOUT, start_new_session=True))
        while any(p.poll() is None for p in processes):
            if time.monotonic()-START > 11*3600:
                raise TimeoutError("Execution budget reached. Checkpoints will be saved; rerun to resume.")
            if any(p.poll() not in (None, 0) for p in processes):
                raise RuntimeError("Evaluation failed; inspect log files in /kaggle/working/gap1-completion")
            if time.monotonic()-last_report > 60:
                print("Elapsed minutes:", round((time.monotonic()-START)/60), flush=True)
                for name, _, _ in commands:
                    print(name, (WORK/(name+".log")).read_text()[-800:], flush=True)
                last_report=time.monotonic()
            if time.monotonic()-last_push > 300:
                push()
                last_push=time.monotonic()
            time.sleep(20)
        if any(p.returncode != 0 for p in processes):
            raise RuntimeError("Evaluation failed; inspect the saved working-directory logs")
    finally:
        for p in processes:
            if p.poll() is None:
                os.killpg(p.pid, signal.SIGTERM)
                try:
                    p.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    os.killpg(p.pid, signal.SIGKILL)
                    p.wait()
        for handle in handles:
            handle.close()


CONDITIONS = ["ka2d_256_standard", "ka2d_1024_slow", "ka2d_1024_standard",
              "ka2d_256_slow", "ka2d_256_fast", "ka2d_1024_fast"]
N_GPU = min(2, torch.cuda.device_count())
MODELS = ["seed0", "seed1", "seed2", "sham_seed0"]
DATA = WORK/"study"
BATCH_SIZE = 4

def command(mode, condition=None, extra=()):
    args=[sys.executable, "-u", "scripts/run_gap2.py", mode, "--out", str(DATA),
          "--workers", str(min(4, os.cpu_count() or 1)), "--batch-size", str(BATCH_SIZE)]
    if condition:
        args += ["--condition", condition]
    return args + list(extra)

try:
    for condition in CONDITIONS:
        print("Condition:", condition, flush=True)
        monitor([(condition+"_hosts", command("hosts",condition), ENV)])
        monitor([(condition+"_requests", command("requests",condition), ENV)])
        push()
        for offset in range(0,len(MODELS),N_GPU):
            jobs=[]
            for gpu, model in enumerate(MODELS[offset:offset+N_GPU]):
                env=dict(ENV, CUDA_VISIBLE_DEVICES=str(gpu))
                jobs.append((condition+"_"+model,
                             command("model",condition,["--model",model,"--device","cuda"]),env))
            if offset==0:
                jobs.append((condition+"_insertion",command("insertion",condition),ENV))
            monitor(jobs)
            push()
        monitor([("report",command("report"),ENV)])
        push()
    state=json.loads((DATA/"study_status.json").read_text())
    assert state["complete"] and len(state["completed_conditions"])==6
    STATUS="complete"
except BaseException:
    STATUS="incomplete"
    raise
finally:
    push()


In [ ]:
import pandas as pd
summary=pd.read_csv(DATA/"placement_summary.csv")
print("Primary denominator: requested sites that were not D2 before editing")
display(summary[summary.definition=="initially_nonD2"])
comparisons=pd.read_csv(DATA/"paired_contrasts.csv")
display(comparisons[(comparisons.definition=="initially_nonD2") &
                    (comparisons.condition=="ka2d_1024_slow")])
print("Intervals resample whole hosts. Seed mean conditions on the three retained models.")
print("A seed0 sham is a same-checkpoint control for seed0; for other seeds the weights also differ.")
print("Placement transfer does not establish structural equivalence or thermal generalization.")
